In [1]:
# Basic imports
import os
import uuid
from tqdm import tqdm

# Third party core
import boto3
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from torchvision import transforms

# Sagemaker imports
import sagemaker
from sagemaker.deserializers import JSONDeserializer
from sagemaker.model_monitor import (
    CronExpressionGenerator,
    DataCaptureConfig,
    DatasetFormat,
    EndpointInput,
    ModelQualityMonitor,
)
from sagemaker.predictor import Predictor
from sagemaker.pytorch import PyTorchModel
from sagemaker.serializers import NumpySerializer, IdentitySerializer

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
%store -r

In [3]:
%store

Stored variables and their in-db values:
bucket                                     -> 'sagemaker-us-east-1-298748835671'
database_name                              -> 'cat_landmarking'
ingest_create_athena_db_passed             -> True
ingestion_completed                        -> True
landmarks_table                            -> 'cat_annotations'
manifest_table                             -> 'image_manifest'
project_prefix                             -> 'cat-landmarks-project'
s3_athena_results_dir                      -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_cats_prefix                   -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_combined_prefix               -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_raw_cats_prefix                         -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_raw_noncats_prefix                      -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_staging_dir                

In [4]:
bucket = bucket
project_prefix = project_prefix
database_name = database_name
manifest_table = manifest_table
s3_staging_dir = s3_staging_dir

s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)

# Monitor image
monitor_image_uri = sagemaker.image_uris.retrieve(framework="model-monitor", region=region)

In [5]:
# Endpoint
endpoint_id = uuid.uuid4()
kp_endpoint_name = f"keypoint-regression-{endpoint_id}"
kp_model_url = "s3://sagemaker-us-east-1-298748835671/pytorch-inference-2026-02-22-05-26-04-291/pipelines-e4q7ozpjg8bf-KeypointRegisterMode-8drrxoQ8MM/output/model.tar.gz"
kp_pipeline_path = "KeypointPipeline/e4q7ozpjg8bf/KeypointPreprocessingEval/output/test"
annotations_key = f"{kp_pipeline_path}/annotations.parquet"

cls_endpoint_name = f"cls-{endpoint_id}"
cls_model_url = "s3://sagemaker-us-east-1-298748835671/cat-landmarks-project/models/artifacts/model_cls_1ed1fb2a-aad1-4019-a688-ecfa2c11da29.tar.gz"

# Creating Keypoint model reference
kp_model = PyTorchModel(
    model_data=kp_model_url,
    role=role,
    framework_version="2.0",
    py_version="py310",
    source_dir="src/keypoint_regression",
    entry_point="inference.py",
    sagemaker_session=sagemaker_session,
)

# Creating Classification model reference
cls_model = PyTorchModel(
    model_data=cls_model_url,
    role=role,
    framework_version="2.0",
    py_version="py310",
    source_dir="src/classification",
    entry_point="inference.py",
    sagemaker_session=sagemaker_session,
)

## Model Monitor set up

In [6]:
data_capture_prefix = f"{project_prefix}/datacapture/{endpoint_id}"
s3_capture_upload_path = f"s3://{bucket}/{data_capture_prefix}"

# Keypoint model configs
kp_data_capture_config = DataCaptureConfig(
    enable_capture=True, 
    sampling_percentage=100, 
    destination_s3_uri=f"{s3_capture_upload_path}/keypoint_regressor"
)

# Deploying Keypoint model
kp_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=kp_endpoint_name,
    data_capture_config=kp_data_capture_config,
)

--------!

In [7]:
# Classifier configs
cls_data_capture_config = DataCaptureConfig(
    enable_capture=True, 
    sampling_percentage=100, 
    destination_s3_uri=f"{s3_capture_upload_path}/classifier"
)

# Deploying Classification model
cls_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=cls_endpoint_name,
    data_capture_config=cls_data_capture_config,
)

--------!

In [8]:
# Keypoint predictor
kp_predictor = Predictor(
    endpoint_name=kp_endpoint_name, 
    sagemaker_session=sagemaker_session,
    serializer=NumpySerializer(),
    deserializer=JSONDeserializer(),
)

# Classification predictor
cls_predictor = Predictor(
    endpoint_name=cls_endpoint_name, 
    sagemaker_session=sagemaker_session,
    serializer=IdentitySerializer("image/jpeg"),
    deserializer=JSONDeserializer(),
)

# Creating Baseline Job
## Keypoint model baselining

In [9]:
baselining_path = Path("./data/baselining")
if not baselining_path.exists():
    os.makedirs(baselining_path)

ANNOTATIONS  = "./data/baselining/annotations.pqt"
NUM_IMAGES   = 100 

KP_NAMES = [
    "left_eye", "right_eye", "mouth",
    "left_ear_1", "left_ear_2", "left_ear_3",
    "right_ear_1", "right_ear_2", "right_ear_3",
]

KP_COLS_NORM = [f"{kp}_{axis}_norm" for kp in KP_NAMES for axis in ("x", "y")]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

TARGET_W, TARGET_H = 224, 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

s3.download_file(bucket, annotations_key, ANNOTATIONS)
df = pd.read_parquet(ANNOTATIONS)
df.head()
sample = df.sample(n=NUM_IMAGES)

In [55]:
# Generate predictions for baselining job     
raw_results = []
baselining_df = []

for _, row in sample.iterrows():
    # Load and resize image from S3
    key = f"{kp_pipeline_path}/{row["image"]}"
    obj = s3.get_object(Bucket=bucket, Key=key)
    img = Image.open(obj["Body"]).convert("RGB")
    w0, h0 = img.size
    img_resized = img.resize((TARGET_W, TARGET_H), Image.BILINEAR)

    # Run inference
    tensor = transform(img_resized).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = kp_predictor.predict(tensor)
    
    # Compute MSE for this image
    gt = row.drop("image")
    
    pred = [{f"{key}_{k}_norm_pred": pred[key][k] for k in pred[key]} for key in pred.keys()]
    pred = pd.Series({k: v for d in pred for k, v in d.items()})
    
    mse = np.mean((pred.to_numpy() - gt.to_numpy()) ** 2)
    baselining_df.append(mse)
    raw_results.append({**dict(gt), **dict(pred)})
    
raw_results = pd.DataFrame(raw_results)
baselining_df = pd.DataFrame({"prediction": baselining_df})
baselining_df["label"] = baselining_df["prediction"].apply(lambda _: 0)
baselining_df.head()

,prediction,label
0,0.168931,0
1,0.119039,0
2,0.116692,0
3,0.125559,0
4,0.207456,0


In [11]:
local_baseline_path = "./data/baselining/baseline_with_predictions.csv"
baselining_df.to_csv(local_baseline_path)

s3_baselining_upload_path = f"{data_capture_prefix}/baseline/baseline_with_predictions.csv"
s3.upload_file(
    local_baseline_path, 
    bucket,
    s3_baselining_upload_path
    )

## Classification model baselining

In [12]:
# Downloading training records
os.makedirs("data/processed/combined/training_manifests/", exist_ok=True)

local_training_manifest_path = "data/processed/combined/training_manifests/training_manifest_021626.pqt"
training_manifest_path = f"{project_prefix}/{local_training_manifest_path}"

s3.download_file(
	bucket,
 	training_manifest_path,
	local_training_manifest_path
)

training_manifest = pd.read_parquet(local_training_manifest_path)
training_manifest.head()

,remote_path,label,split
0,s3://sagemaker-us-east-1-298748835671/cat-land...,0,train
1,s3://sagemaker-us-east-1-298748835671/cat-land...,0,validation
2,s3://sagemaker-us-east-1-298748835671/cat-land...,0,train
3,s3://sagemaker-us-east-1-298748835671/cat-land...,0,train
4,s3://sagemaker-us-east-1-298748835671/cat-land...,0,train


In [52]:
label_map = dict(zip(training_manifest["remote_path"], training_manifest["label"]))
test_uris = training_manifest.loc[training_manifest["split"] == "test", "remote_path"]

rows = []
for uri in tqdm(test_uris, desc="Performing Batch Inference", unit="image"):
    label = label_map[uri]
    key = uri.replace(f"s3://{bucket}/", "")
    
    try:
        payload = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
        result = cls_predictor.predict(payload)
        pred = result["predicted_class"]
        rows.append({"uri": uri, "label": label, "pred": pred, "error": None})
    except Exception as e:
        rows.append({"uri": uri, "label": label, "pred": None, "error": str(e)})

test_df = pd.DataFrame(rows)
print(f"Completed: {test_df['pred'].notna().sum()}/{len(test_df)} successful")

Performing Batch Inference: 100%|██████████| 2522/2522 [03:37<00:00, 11.59image/s]

Completed: 2522/2522 successful


In [36]:
cls_local_baseline_path = "./data/baselining/cls_baseline_with_predictions.csv"
test_df[["label","pred"]].to_csv(cls_local_baseline_path, index=False)

s3_cls_baselining_upload_path = f"{data_capture_prefix}/baseline/cls_baseline_with_predictions.csv"
s3.upload_file(
    cls_local_baseline_path, 
    bucket,
    s3_cls_baselining_upload_path
    )

# Creating Model Quality Monitors

## Keypoint Model baselining

In [25]:
kp_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

kp_baseline_job_name = f"keypoint-baseline-job-{endpoint_id}"
job = kp_monitor.suggest_baseline(
    job_name = kp_baseline_job_name,
    baseline_dataset=f"s3://{bucket}/{s3_baselining_upload_path}",
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="Regression",
    inference_attribute="prediction",
    ground_truth_attribute="label",
    output_s3_uri=f"s3://{bucket}/{project_prefix}/{data_capture_prefix}/baseline/results",
)

job.wait(logs=False)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name keypoint-baseline-job-96cebf92-cacd-4118-852e-39835de11537


...........................................................!

In [26]:
kp_baseline_job = kp_monitor.latest_baselining_job

In [27]:
pd.DataFrame(kp_baseline_job.baseline_statistics().body_dict["regression_metrics"]).T

,value,standard_deviation
mae,0.094578,NaN
mse,0.013022,NaN
rmse,0.114115,NaN
r2,-Infinity,None


In [28]:
pd.DataFrame(kp_baseline_job.suggested_constraints().body_dict["regression_constraints"]).T

,threshold,comparison_operator
mae,0.094578,GreaterThanThreshold
mse,0.013022,GreaterThanThreshold
rmse,0.114115,GreaterThanThreshold
r2,-Infinity,LessThanThreshold


## Classifier baselining

In [41]:
cls_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)

cls_baseline_job_name = f"cls-baseline-job-{endpoint_id}1"
job = cls_monitor.suggest_baseline(
    job_name = cls_baseline_job_name,
    baseline_dataset=f"s3://{bucket}/{s3_cls_baselining_upload_path}",
    dataset_format=DatasetFormat.csv(header=True),
    problem_type="BinaryClassification",
    inference_attribute="pred",
    ground_truth_attribute="label",
    output_s3_uri=f"s3://{bucket}/{project_prefix}/{data_capture_prefix}/baseline/results",
)

job.wait(logs=False)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name cls-baseline-job-96cebf92-cacd-4118-852e-39835de115371


...........................................................!

In [42]:
cls_baseline_job = cls_monitor.latest_baselining_job

In [45]:
pd.DataFrame(cls_baseline_job.suggested_constraints().body_dict["binary_classification_constraints"]).T

,threshold,comparison_operator
recall,0.964965,LessThanThreshold
precision,0.98167,LessThanThreshold
accuracy,0.978985,LessThanThreshold
true_positive_rate,0.964965,LessThanThreshold
true_negative_rate,0.988181,LessThanThreshold
false_positive_rate,0.011819,GreaterThanThreshold
false_negative_rate,0.035035,GreaterThanThreshold
f0_5,0.978283,LessThanThreshold
f1,0.973246,LessThanThreshold
f2,0.96826,LessThanThreshold


# Setting up Monitoring Schedule

In [ ]:
kp_monitor.create_monitoring_schedule(
    monitor_schedule_name="keypoint-model-quality-schedule",
    endpoint_input=EndpointInput(
        endpoint_name=kp_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="mse",
    ),
    ground_truth_input=f"s3://{bucket}/{project_prefix}/ground-truth/",
    problem_type="Regression",
    output_s3_uri=f"s3://{bucket}/{project_prefix}/monitoring/results",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)

In [ ]:
cls_monitor.create_monitoring_schedule(
    monitor_schedule_name="cls-model-quality-schedule",
    endpoint_input=EndpointInput(
        endpoint_name=cls_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="mse",
    ),
    ground_truth_input=f"s3://{bucket}/{project_prefix}/ground-truth/",
    problem_type="BinaryClassification",
    output_s3_uri=f"s3://{bucket}/{project_prefix}/monitoring/results",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: cls-model-quality-schedule-1


# Creating CloudWatch Monitor

In [50]:
cw_client = boto3.client("cloudwatch", region_name=region)

cw_client.put_metric_alarm(
    AlarmName="KeypointModel-MSE-Violation",
    AlarmDescription="Triggers when model MSE violates baseline constraints",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    MetricName="constraint_violations",
    Dimensions=[
        {"Name": "Endpoint", "Value": kp_endpoint_name},
        {"Name": "MonitoringSchedule", "Value": "keypoint-model-quality-schedule"},
    ],
    Statistic="Sum",
    Period=3600,   
    EvaluationPeriods=1,
    Threshold=1,        
    ComparisonOperator="GreaterThanOrEqualToThreshold",
    TreatMissingData="notBreaching",
)

{'ResponseMetadata': {'RequestId': '0700d949-286e-4545-aa10-c777d99ce29e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '0700d949-286e-4545-aa10-c777d99ce29e',
   'content-type': 'text/xml',
   'content-length': '214',
   'date': 'Sun, 22 Feb 2026 23:33:37 GMT'},
  'RetryAttempts': 0}}

In [51]:
cw_client = boto3.client("cloudwatch", region_name=region)

cw_client.put_metric_alarm(
    AlarmName="ClsModel-Accuracy-Violation",
    AlarmDescription="Triggers when model Accuracy violates baseline constraints",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    MetricName="constraint_violations",
    Dimensions=[
        {"Name": "Endpoint", "Value": cls_endpoint_name},
        {"Name": "MonitoringSchedule", "Value": "cls-model-quality-schedule"},
    ],
    Statistic="Sum",
    Period=3600,   
    EvaluationPeriods=1,
    Threshold=1,        
    ComparisonOperator="LessThanThreshold",
    TreatMissingData="notBreaching",
)

{'ResponseMetadata': {'RequestId': 'f3b5b9d4-7e6d-4039-a8c1-b35de8d21570',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'f3b5b9d4-7e6d-4039-a8c1-b35de8d21570',
   'content-type': 'text/xml',
   'content-length': '214',
   'date': 'Sun, 22 Feb 2026 23:34:19 GMT'},
  'RetryAttempts': 0}}

# Deleteing endpoint

In [ ]:
cw_client.delete_alarms(AlarmNames=["KeypointModel-MSE-Violation"])
cw_client.delete_alarms(AlarmNames=["ClsModel-Accuracy-Violation"])

kp_monitor.stop_monitoring_schedule()
kp_monitor.delete_monitoring_schedule()

cls_monitor.stop_monitoring_schedule()
cls_monitor.delete_monitoring_schedule()

In [ ]:
sagemaker_client.delete_endpoint(EndpointName=kp_endpoint_name)
sagemaker_client.delete_endpoint(EndpointName=cls_endpoint_name)